In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Стандартні налаштування та параметри конфігурації транспілятора


<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблено з використанням наступних залежностей.
Рекомендуємо використовувати ці версії або новіші.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Абстрактні схеми потребують транспіляції, оскільки QPU підтримують обмежений набір базисних Gate і не можуть виконувати довільні операції. Функція транспілятора — перетворювати довільні схеми так, щоб вони могли виконуватися на конкретному QPU. Це досягається шляхом перекладу схем до підтримуваних базисних Gate і додавання SWAP Gate за потреби, щоб зв'язність схеми відповідала зв'язності QPU.

Як пояснено в розділі [Транспіляція з менеджерами проходів](transpile-with-pass-managers), ти можеш створити [менеджер проходів](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) за допомогою функції [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) та передати схему або список схем до його методу [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) для транспіляції. Можна викликати `generate_preset_pass_manager`, передаючи лише рівень оптимізації та backend, використовуючи стандартні значення для всіх інших параметрів, або ж передати додаткові аргументи для тонкого налаштування транспіляції.

## Базове використання без параметрів
У цьому прикладі ми передаємо схему та цільовий QPU транспілятору, не вказуючи жодних додаткових параметрів.

Створи схему та перегляни результат:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Транспілюй схему та перегляни результат:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Усі доступні параметри
Нижче наведено всі доступні параметри функції [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Існує два класи аргументів: ті, що описують ціль компіляції, і ті, що впливають на роботу транспілятора.

Усі параметри, крім `optimization_level`, є необов'язковими. Для отримання повної інформації дивись [документацію Transpiler API](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) — рівень оптимізації схем. Ціле число в діапазоні (0–3). Вищі рівні генерують більш оптимізовані схеми ціною збільшеного часу транспіляції. Докладніше дивись у розділі [Встановлення рівня оптимізації транспілятора](set-optimization).

### Параметри для опису цілі компіляції:
Ці аргументи описують цільовий QPU для виконання схем, включаючи такі відомості, як карта зв'язності QPU (що описує з'єднання між Qubit), базисні Gate, підтримувані QPU, та частоти помилок Gate.

Багато з цих параметрів докладно описано в розділі [Часто використовувані параметри транспіляції](common-parameters).

<details>
  <summary>
    **Параметри QPU (`Backend`)**
  </summary>

**Параметри Backend** — якщо ти вказуєш `backend`, не потрібно вказувати `target` або інші параметри backend. Аналогічно, якщо ти вказуєш `target`, не потрібно вказувати `backend` або інші параметри backend.
- `backend` (Backend) — якщо вказано, транспілятор компілює вхідну схему для цього пристрою. Якщо встановлено будь-який інший параметр, що впливає на ці налаштування, наприклад `coupling_map`, він замінює налаштування з `backend`.
- `target` (Target) — ціль транспілятора для backend. Зазвичай це вказується як частина аргументу backend, але якщо ти вручну побудував об'єкт Target, можна вказати його тут. Це замінює ціль із `backend`.
- `backend_properties` (BackendProperties) — властивості QPU, включаючи інформацію про помилки Gate, помилки зчитування, часи когерентності Qubit тощо. Знайди QPU, що надає цю інформацію, виконавши `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) — необов'язкове апаратне обмеження на роздільну здатність часу виконання інструкцій. Ця інформація надається конфігурацією QPU. Якщо QPU не має жодних обмежень на розподіл часу виконання інструкцій, `timing_constraints` дорівнює `None` і корекція не виконується. QPU може повідомляти про такі обмеження:
    - `granularity`: ціле число, що представляє мінімальну роздільну здатність імпульсного Gate у одиницях dt. Тривалість користувацького імпульсного Gate повинна бути кратна цьому значенню.
    - `min_length`: ціле число, що представляє мінімальну довжину імпульсного Gate у одиницях dt. Користувацький імпульсний Gate повинен бути довшим за це значення.
    - `pulse_alignment`: ціле число, що представляє роздільну здатність часу для початку виконання Gate інструкції. Інструкції Gate повинні починатися у момент, кратний цьому значенню.
    - `acquire_alignment`: ціле число, що представляє роздільну здатність часу для початку виконання інструкції вимірювання. Інструкція вимірювання повинна починатися у момент, кратний цьому значенню.
</details>

<details>
  <summary>
    **Параметри розміщення та топології**
  </summary>

- `basis_gates` (List[str] | None) — список назв базисних Gate для розгортання. Наприклад ['u1', 'u2', 'u3', 'cx']. Якщо `None`, розгортання не виконується.
- `coupling_map` (CouplingMap | List[List[int]]) — спрямована карта зв'язності (можливо, користувацька) для відображення. Якщо карта зв'язності симетрична, потрібно вказати обидва напрямки. Підтримуються такі формати:
    - Екземпляр CouplingMap
    - List — повинен бути заданий як матриця суміжності, де кожен запис визначає всі дозволені спрямовані двокубітові взаємодії на QPU. Наприклад: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) — відображення операцій схеми до імпульсних розкладів. Якщо `None`, використовується `instruction_schedule_map` QPU.
</details>

### Параметри, що впливають на роботу транспілятора
Ці параметри впливають на конкретні етапи транспіляції. Деякі з них можуть впливати на кілька етапів, але для простоти вони перераховані лише під одним. Якщо ти вказуєш аргумент, наприклад `initial_layout` для Qubit, які хочеш використовувати, це значення замінює всі проходи, що могли б його змінити. Іншими словами, транспілятор не змінить нічого, що ти вказуєш вручну. Для отримання інформації про конкретні етапи дивись [Етапи транспілятора](transpiler-stages).

<details>
  <summary>
    **Етап ініціалізації**
  </summary>

- `hls_config` (HLSConfig) — необов'язковий клас конфігурації `HLSConfig`, що передається безпосередньо до проходу трансформації `HighLevelSynthesis`. Цей клас конфігурації дозволяє вказувати списки алгоритмів синтезу та їхні параметри для різних об'єктів високого рівня.
- `init_method` (str) — назва плагіна для використання на етапі ініціалізації. За замовчуванням зовнішній плагін не використовується. Список встановлених плагінів можна переглянути, виконавши `list_stage_plugins()` з `init` як аргументом назви етапу.
- `unitary_synthesis_method` (str) — назва методу унітарного синтезу. За замовчуванням використовується `default`. Список встановлених плагінів можна переглянути, виконавши `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) — необов'язковий словник конфігурації, що передається безпосередньо до плагіна унітарного синтезу. За замовчуванням це налаштування не має ефекту, оскільки стандартний метод унітарного синтезу не приймає користувацьку конфігурацію. Застосовувати користувацьку конфігурацію потрібно лише тоді, коли плагін унітарного синтезу вказано через аргумент `unitary_synthesis`. Оскільки це унікально для кожного плагіна, зверни увагу на документацію плагіна, щоб дізнатися, як використовувати цю опцію.
</details>

<details>
  <summary>
    **Етап розміщення**
  </summary>

- `initial_layout` (Layout | Dict | List) — початкове розміщення віртуальних Qubit на фізичних Qubit. Якщо це розміщення робить схему сумісною з обмеженнями `coupling_map`, воно буде використане. Кінцеве розміщення не гарантується таким самим, оскільки транспілятор може переставляти Qubit через SWAP або інші засоби. Для повної інформації дивись [розділ про початкове розміщення.](common-parameters#initial-layout)
- `layout_method` (str) — назва проходу вибору розміщення (`default`, `dense`, `sabre`, `trivial`). Це також може бути назва зовнішнього плагіна для етапу розміщення. Список встановлених плагінів можна переглянути, виконавши `list_stage_plugins()` з `layout` як аргументом `stage_name`. Стандартне значення — `sabre`.
</details>

<details>
  <summary>
    **Етап маршрутизації**
  </summary>

- `routing_method` (str) — назва проходу маршрутизації (`basic`, `lookahead`, `default`, `sabre` або `none`). Це також може бути назва зовнішнього плагіна для етапу маршрутизації. Список встановлених плагінів можна переглянути, виконавши `list_stage_plugins()` з `routing` як аргументом `stage_name`. Стандартне значення — `sabre`.
</details>

<details>
  <summary>
    **Етап трансляції**
  </summary>

- `translation_method` (str) — назва проходу трансляції (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Це також може бути назва зовнішнього плагіна для етапу трансляції. Список встановлених плагінів можна переглянути, виконавши `list_stage_plugins()` з `translation` як аргументом `stage_name`. Стандартне значення — `translator`.
</details>

<details>
  <summary>
    **Етап оптимізації**
  </summary>

- `approximation_degree` (float, у діапазоні 0–1 | None) — евристичний параметр для апроксимації схеми (1.0 = без апроксимації, 0.0 = максимальна апроксимація). Стандартне значення — 1.0. Вказання `None` встановлює ступінь апроксимації відповідно до зафіксованої частоти помилок. Докладніше дивись у [розділі про ступінь апроксимації](common-parameters#approx-degree).
- `optimization_method` (str) — назва плагіна для використання на етапі оптимізації. За замовчуванням зовнішній плагін не використовується. Список встановлених плагінів можна переглянути, виконавши `list_stage_plugins()` з `optimization` як аргументом `stage_name`.
</details>

<details>
  <summary>
    **Етап планування**
  </summary>

- `scheduling_method` (str) — назва проходу планування. Це також може бути назва зовнішнього плагіна для етапу планування. Список встановлених плагінів можна переглянути, виконавши `list_stage_plugins()` з `scheduling` як аргументом `stage_name`.
  * 'as_soon_as_possible': планувати інструкції жадібно, якомога раніше на ресурсі Qubit (псевдонім: `asap`).
  * 'as_late_as_possible': планувати інструкції пізніше, тобто утримувати Qubit в основному стані, коли це можливо (псевдонім: `alap`). Це стандартне значення.
</details>

<details>
  <summary>
    **Виконання транспілятора**
  </summary>

- `seed_transpiler` (int) — встановлює випадкові початкові числа для стохастичних частин транспілятора.
</details>

Якщо ти не вказуєш жодного з наведених вище параметрів, використовуються такі стандартні значення. Для отримання додаткової інформації дивись [сторінку довідника API](../api/qiskit/transpiler_preset) методу:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>